### Certain genres from past 10 years will be more popular than others and better for our artist profiles

- artist_genre - use current data
    - standard statistics, averages, describe
    - song popularity

In [23]:
import pandas as pd
import numpy as np
import ast
# for syntax analysis

# 1. Extract release year and filter for the last 10 years (2016 - 2026)
df['release_year'] = pd.to_datetime(df['album_release_date'], errors='coerce').dt.year

df_last_10_years = df[df['release_year'] >= 2016].copy()

print(f"Original dataset rows: {len(df)}")
print(f"Filtered dataset rows (last 10 years, >= 2016: {len(df_last_10_years)}")
print("-" * 60)

# 1. Helper function to safely parse genre strings like "['rap', 'hip hop']" or handle missing or special values: '[]'
def parse_genres(genre_val):
    if pd.isna(genre_val) or genre_val == 'N/A' or genre_val == '[]':
        return ['unknown']
    if isinstance(genre_val, list):
        return genre_val if genre_val else ['unknown']
    try:
        # Try evaluating string representation of list
        parsed = ast.literal_eval(genre_val)
        if isinstance(parsed, list) and len(parsed) > 0:
            return parsed
    except:
        pass
    # Fallback for simple comma-separated strings if any
    cleaned = [g.strip(" '\"[]") for g in str(genre_val).split(',') if g.strip()]
    return cleaned if cleaned else ['unknown']

# 2. Create a temporary exploded dataframe for genre analysis
df_temp = df_last_10_years.copy()
df_temp['parsed_genres'] = df_temp['artist_genres'].apply(parse_genres)

# Explode rows so each genre gets its own row per track (supports multiple genres per song)
df_exploded = df_temp.explode('parsed_genres')

print(f"Total track-genre rows after exploding: {len(df_exploded)}")
print("-" * 60)

# 3. Standard statistics and averages for Track Popularity per Genre
print("--- Track Popularity Statistics by Genre ---")
genre_popularity_stats = df_exploded.groupby('parsed_genres')['track_popularity'].agg(
    track_count='count',
    mean_popularity='mean',
    median_popularity='median',
    min_popularity='min',
    max_popularity='max',
    std_popularity='std'
).reset_index()

# Sort by mean popularity descending and filter out extremely rare genres if needed
genre_popularity_stats = genre_popularity_stats.sort_values(by='mean_popularity', ascending=False)

# Display top 15 genres by average track popularity
display(genre_popularity_stats.head(15))

print("-" * 60)
print("--- Summary Describe for Track Popularity overall ---")
display(df['track_popularity'].describe())

Original dataset rows: 8582
Filtered dataset rows (last 10 years, >= 2016: 5179
------------------------------------------------------------
Total track-genre rows after exploding: 8201
------------------------------------------------------------
--- Track Popularity Statistics by Genre ---


,parsed_genres,track_count,mean_popularity,median_popularity,min_popularity,max_popularity,std_popularity
251,pop soul,3,90.333333,90.0,86,95,4.509250
318,trap funk,1,85.000000,85.0,85,85,NaN
134,funk consciente,1,85.000000,85.0,85,85,NaN
277,reggaeton chileno,1,84.000000,84.0,84,84,NaN
7,afro tech,1,82.000000,82.0,82,82,NaN
114,electro corridos,5,81.000000,83.0,71,87,6.928203
123,europop,1,80.000000,80.0,80,80,NaN
91,dembow belico,2,80.000000,80.0,77,83,4.242641
292,sierreño,6,79.333333,79.0,70,87,7.284687
178,indonesian indie,1,79.000000,79.0,79,79,NaN


------------------------------------------------------------
--- Summary Describe for Track Popularity overall ---


count    8582.000000
mean       52.356211
std        23.816076
min         0.000000
25%        39.000000
50%        58.000000
75%        71.000000
max        99.000000
Name: track_popularity, dtype: float64

In [25]:

# 5. Most frequent genres (sorted by track_count descending)
sorted_by_frequency = genre_popularity_stats.sort_values(by='track_count', ascending=False)

print("--- Top Genres Sorted by Frequency (Track Count - Last 10 Years) ---")
display(sorted_by_frequency.head(15))

print("-" * 60)

# 6. Most popular genres with a minimum occurrence threshold (at least x tracks)
min_count_threshold = 100 

popular_frequent_genres = genre_popularity_stats[genre_popularity_stats['track_count'] >= min_count_threshold]
popular_frequent_genres = popular_frequent_genres.sort_values(by='mean_popularity', ascending=False)

print(f"--- Top Popular Genres (appearing at least {min_count_threshold} times - Last 10 Years) ---")
display(popular_frequent_genres.head(15))

--- Top Genres Sorted by Frequency (Track Count - Last 10 Years) ---


,parsed_genres,track_count,mean_popularity,median_popularity,min_popularity,max_popularity,std_popularity
328,unknown,2233,48.725929,54.0,0,94,24.741932
247,pop,647,63.570325,66.0,0,97,17.085133
76,country,443,59.275395,63.0,0,97,17.581253
171,indie,268,62.910448,66.0,24,97,17.457262
125,folk,252,62.055556,65.0,24,97,17.590203
164,hip hop,231,60.441558,63.0,0,91,19.380286
271,rap,162,67.543210,68.5,0,85,12.925058
16,alternative pop,161,66.583851,71.0,2,94,15.899905
301,soundtrack,159,34.465409,38.0,0,71,16.673628
298,soft pop,138,52.318841,60.0,0,88,24.984663


------------------------------------------------------------
--- Top Popular Genres (appearing at least 100 times - Last 10 Years) ---


,parsed_genres,track_count,mean_popularity,median_popularity,min_popularity,max_popularity,std_popularity
271,rap,162,67.543210,68.5,0,85,12.925058
16,alternative pop,161,66.583851,71.0,2,94,15.899905
247,pop,647,63.570325,66.0,0,97,17.085133
307,synthpop,125,62.992000,66.0,1,91,18.717551
171,indie,268,62.910448,66.0,24,97,17.457262
125,folk,252,62.055556,65.0,24,97,17.590203
164,hip hop,231,60.441558,63.0,0,91,19.380286
76,country,443,59.275395,63.0,0,97,17.581253
17,alternative r&b,118,58.415254,63.0,1,91,20.442676
29,art pop,107,53.981308,58.0,19,91,18.524324
